In [3]:
import sys
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LassoCV
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

current_dir = os.getcwd()
sys.path.append(current_dir)
if 'Tuan04' not in current_dir:
    sys.path.append(os.path.join(current_dir, 'Tuan04'))

# [LỖI 2, LỖI 3]: Đổi sang thư mục scr và import biến môi trường chung của nhóm
from scr.evaluate import evaluate_model
from scr.models import make_models, K_FEATURES, N_SPLITS, RANDOM_STATE

print("Đang nạp dữ liệu...")
X_train = pd.read_csv('data/processed/X_train.csv')
X_test = pd.read_csv('data/processed/X_test.csv')
# [LỖI 5]: Thêm .squeeze('columns') để chuyển DataFrame thành Series chuẩn
y_train = pd.read_csv('data/processed/y_train.csv').squeeze('columns')
y_test = pd.read_csv('data/processed/y_test.csv').squeeze('columns')

# Lấy các model đã được tune từ TV3/TV4 thay vì model mặc định
models = make_models()

def get_embedded_lasso(model):
    # [LỖI 4]: Thêm max_features=K_FEATURES và threshold=-np.inf
    selector = SelectFromModel(
        LassoCV(cv=5, random_state=RANDOM_STATE, n_jobs=-1),
        max_features=K_FEATURES,
        threshold=-np.inf
    )
    return Pipeline([('feature_selection', selector), ('model', model)])

def get_embedded_rf(model):
    rf = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    selector = SelectFromModel(rf, threshold=-np.inf, max_features=K_FEATURES)
    return Pipeline([('feature_selection', selector), ('model', model)])

results = []
fitted_pipelines_lasso = {}
fitted_pipelines_rf = {}

print("Đang chạy Embedded Lasso...")
for name, model in models.items():
    pipe_lasso = get_embedded_lasso(model)
    # [LỖI 5, LỖI 6]: Gọi evaluate_model với N_SPLITS, seed và return_model=True
    res, fitted_model = evaluate_model(
        'embedded_lasso', name, pipe_lasso, X_train, y_train, X_test, y_test,
        n_splits=N_SPLITS, seed=RANDOM_STATE, return_model=True
    )
    results.append(res)
    fitted_pipelines_lasso[name] = fitted_model # Tránh NotFittedError

print("Đang chạy Embedded Random Forest...")
for name, model in models.items():
    pipe_rf = get_embedded_rf(model)
    res, fitted_model = evaluate_model(
        'embedded_rf', name, pipe_rf, X_train, y_train, X_test, y_test,
        n_splits=N_SPLITS, seed=RANDOM_STATE, return_model=True
    )
    results.append(res)
    fitted_pipelines_rf[name] = fitted_model

# Lưu file kết quả của riêng TV5
df_embedded = pd.DataFrame(results)
df_embedded.to_csv('data/processed/fs_results_embedded.csv', index=False)

# =======================================================
# [LỖI 8]: GỘP DATA BÀI CỦA TV3 VÀ TV4 ĐỂ VẼ HÌNH CHUNG
# =======================================================
print("Đang vẽ biểu đồ so sánh tổng hợp...")
df_baseline = pd.read_csv('data/processed/baseline_results.csv')
df_fw = pd.read_csv('data/processed/fs_results_filter_wrapper.csv')
df_all = pd.concat([df_baseline, df_fw, df_embedded], ignore_index=True)

# Hình 1: RMSE
plt.figure(figsize=(12, 6))
sns.barplot(data=df_all, x='technique', y='cv_log_rmse', hue='model')
plt.title('CV Log RMSE Comparison (All Techniques)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/rmse_by_technique.png', dpi=150)
plt.close()

# Hình 2: Delta Heatmap
pivot_all = df_all.pivot(index='technique', columns='model', values='cv_log_rmse')
baseline_series = pivot_all.loc['baseline']
delta_heatmap = (pivot_all - baseline_series) / baseline_series * 100

plt.figure(figsize=(10, 6))
sns.heatmap(delta_heatmap, annot=True, cmap='RdYlGn_r', center=0, fmt='.2f')
plt.title('Percentage Change in RMSE vs Baseline (%)')
plt.tight_layout()
plt.savefig('figures/delta_heatmap.png', dpi=150)
plt.close()

# Hình 3: Time
plt.figure(figsize=(12, 6))
sns.barplot(data=df_all, x='technique', y='train_seconds', hue='model')
plt.title('Training Time Comparison (seconds)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('figures/time_comparison.png', dpi=150)
plt.close()

# =======================================================
# [LỖI 7]: ĐẾM CHUNG FEATURE VỚI BÀI CỦA TV4 (Từ 1 đến 5)
# =======================================================
# Trích xuất feature từ mô hình cuối (XGBoost) đã được fit an toàn
lasso_sel = fitted_pipelines_lasso['XGBoost'].named_steps['feature_selection'].get_support()
rf_sel = fitted_pipelines_rf['XGBoost'].named_steps['feature_selection'].get_support()

tv5_features = X_train.columns[lasso_sel].tolist() + X_train.columns[rf_sel].tolist()

# Đọc kết quả feature của TV4
df_tv4_features = pd.read_csv('data/processed/selected_features_filter_wrapper.csv')

# TỰ ĐỘNG TÌM TÊN CỘT: Nếu TV4 không dùng tên 'Feature' thì tự động lấy cột đầu tiên (index 0)
feature_col = 'Feature' if 'Feature' in df_tv4_features.columns else df_tv4_features.columns[0]
count_col = 'Count' if 'Count' in df_tv4_features.columns else None

tv4_features_list = []
if count_col is not None:
    for _, row in df_tv4_features.iterrows():
        tv4_features_list.extend([row[feature_col]] * int(row[count_col]))
else:
    tv4_features_list = df_tv4_features[feature_col].tolist()

all_selected_features = tv5_features + tv4_features_list

from collections import Counter
feature_counts = Counter(all_selected_features)
df_features = pd.DataFrame(feature_counts.items(), columns=['Feature', 'Count']).sort_values('Count', ascending=False)

# Lưu lại file chuẩn của toàn dự án
df_features.to_csv('data/processed/selected_features.csv', index=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=df_features.head(15), x='Count', y='Feature', palette='viridis')
plt.title('Top Features Agreement Across All 5 Techniques')
plt.tight_layout()
plt.savefig('figures/feature_agreement.png', dpi=150)
plt.close()

print("HOÀN TẤT ĐẾM FEATURE VÀ XUẤT HÌNH!")

Đang nạp dữ liệu...
Đang chạy Embedded Lasso...
Đang chạy Embedded Random Forest...
Đang vẽ biểu đồ so sánh tổng hợp...
HOÀN TẤT ĐẾM FEATURE VÀ XUẤT HÌNH!
